# Diffusion-Regularized PINNs Demo (Gaussian Initial Condition)

This notebook demonstrates the comparison between:
- **Poisson**: Poisson-only inverse PINN (P-iPINN in paper)
- **Coupled**: Poisson + Diffusion-Decay regularized PINN (PD-iPINN in paper)

**Initial condition**: Gaussian charge distribution (realistic KPFM scenario)

**Runtime**: ~5 minutes on Colab GPU

In [ ]:
# =============================================================================
# IMPORTANT: Set backend BEFORE importing deepxde
# =============================================================================
import os
os.environ['DDE_BACKEND'] = 'tensorflow.compat.v1'

# Install dependencies (Colab)
!pip install deepxde==1.14.0 --quiet

In [ ]:
# =============================================================================
# Imports
# =============================================================================
import deepxde as dde
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.special import erf
import warnings
warnings.filterwarnings('ignore')

print(f"DeepXDE: {dde.__version__}")
print(f"Backend: {dde.backend.backend_name}")

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

# Random seed for reproducibility
SEED = 42

# Noise level (0.0 to 0.5)
NOISE_LEVEL = 0.5  # 50% noise

# Grid resolution (fixed for Gaussian)
NUM_X = 101  # Spatial grid points
NUM_T = 101  # Temporal grid points

# Training settings
EPOCHS = 5000
DISPLAY_EVERY = 1000
NUM_DOMAIN = 2000

# Physical parameters
EPS = 1.0        # Permittivity
D_COEFF = 0.01   # Diffusion coefficient [m²/s]
K_COEFF = 0.5    # Decay coefficient [1/s]

# Gaussian parameters
Q0 = 1.0         # Initial total charge
SIGMA0 = 0.1     # Initial standard deviation

# Domain bounds
X_MIN, X_MAX = -1.0, 1.0
T_MIN, T_MAX = 0.0, 1.0

print(f"Configuration: noise={NOISE_LEVEL*100:.0f}%, grid=({NUM_X}×{NUM_T})")
print(f"Gaussian: Q0={Q0}, σ0={SIGMA0}")
print(f"Physics: D={D_COEFF}, k={K_COEFF}")

In [ ]:
# =============================================================================
# Exact Solutions
# =============================================================================

def generate_gaussian_data(x_grid, t_grid):
    """
    Generate analytical solution for Gaussian initial condition.

    ρ(x,t) = Q(t)/(√(2π)σ(t)) * exp(-x²/(2σ(t)²))
    where:
        σ(t) = √(σ0² + 2Dt)  (diffusion spreading)
        Q(t) = Q0 * exp(-kt)  (decay)

    V(x,t) from quasi-static Poisson equation
    """
    X, T = np.meshgrid(x_grid, t_grid, indexing='ij')  # (Nx, Nt)

    # σ(t) and Q(t)
    sigma_t = np.sqrt(SIGMA0**2 + 2*D_COEFF*T)
    Q_t = Q0 * np.exp(-K_COEFF*T)

    # ρ analytical
    rho_mesh = Q_t / (np.sqrt(2*np.pi) * sigma_t) * np.exp(-X**2 / (2*sigma_t**2))

    # V analytical (Poisson solution)
    z = X / (np.sqrt(2) * sigma_t)
    V_raw = -Q_t / (2*EPS) * (
        X * erf(z) +
        sigma_t * np.sqrt(2/np.pi) * np.exp(-z**2)
    )
    # Mean-zero gauge
    V_mesh = V_raw - np.mean(V_raw, axis=0, keepdims=True)

    return rho_mesh, V_mesh


def add_noise(V_clean, noise_level, seed):
    """Add relative Gaussian noise: V_obs = V_ex + σ·max|V_ex|·η"""
    np.random.seed(seed)
    if noise_level == 0:
        return V_clean.copy()
    V_scale = np.max(np.abs(V_clean))
    return V_clean + noise_level * V_scale * np.random.randn(*V_clean.shape)

In [ ]:
# =============================================================================
# PDE Residuals
# =============================================================================

def pde_poisson_only(X, y):
    """Poisson model: -∂²φ/∂x² - ρ/ε = 0"""
    phi_xx = dde.grad.hessian(y, X, component=0, i=0, j=0)
    rho = y[:, 1:2]
    return -phi_xx - rho / EPS

def pde_coupled(X, y):
    """Coupled model: Poisson + Diffusion-Decay

    1. Poisson: -∂²φ/∂x² - ρ/ε = 0
    2. Diffusion-Decay: ∂ρ/∂t - D∂²ρ/∂x² + kρ = 0
    """
    phi_xx = dde.grad.hessian(y, X, component=0, i=0, j=0)
    rho = y[:, 1:2]
    rho_t = dde.grad.jacobian(y, X, i=1, j=1)
    rho_xx = dde.grad.hessian(y, X, component=1, i=0, j=0)

    res_poisson = -phi_xx - rho / EPS
    res_diffusion = rho_t - D_COEFF * rho_xx + K_COEFF * rho

    return [res_poisson, res_diffusion]

In [ ]:
# =============================================================================
# Training Function
# =============================================================================

def train_model(model_type, phi_obs, XT_data, seed):
    tf.compat.v1.reset_default_graph()
    dde.config.set_random_seed(seed)
    np.random.seed(seed)

    # Geometry
    geom = dde.geometry.Interval(X_MIN, X_MAX)
    timedomain = dde.geometry.TimeDomain(T_MIN, T_MAX)
    geomtime = dde.geometry.GeometryXTime(geom, timedomain)

    # Observation constraint
    observe_phi = dde.icbc.PointSetBC(XT_data, phi_obs, component=0)

    # Model-specific settings
    if model_type == 'Poisson':
        pde_fn = pde_poisson_only
        loss_weights = [1, 1000]
    else:  # Coupled
        pde_fn = pde_coupled
        loss_weights = [1, 1, 1000]

    # Data
    data = dde.data.TimePDE(
        geomtime, pde_fn, [observe_phi],
        num_domain=NUM_DOMAIN, num_boundary=0, num_initial=0,
        anchors=XT_data, num_test=5000
    )

    # Network
    net = dde.nn.PFNN(
        [2, [64, 64], [64, 64], [64, 64], [64, 64], 2],
        "tanh", "Glorot uniform"
    )

    # Compile and train
    model = dde.Model(data, net)
    model.compile("adam", lr=1e-3, loss_weights=loss_weights)

    print(f"\nTraining {model_type}...")
    losshistory, _ = model.train(epochs=EPOCHS, display_every=DISPLAY_EVERY)

    # Predict
    output = model.predict(XT_data)
    phi_pred = output[:, 0:1]
    rho_pred = output[:, 1:2]

    tf.keras.backend.clear_session()

    return phi_pred, rho_pred, losshistory

In [ ]:
# =============================================================================
# Generate Data
# =============================================================================

# Create grid
x_grid = np.linspace(X_MIN, X_MAX, NUM_X)
t_grid = np.linspace(T_MIN, T_MAX, NUM_T)
X_mesh, T_mesh = np.meshgrid(x_grid, t_grid, indexing='ij')
XT_data = np.hstack([X_mesh.flatten()[:, None], T_mesh.flatten()[:, None]])

# Generate Gaussian analytical solution
rho_ex_mesh, phi_ex_mesh = generate_gaussian_data(x_grid, t_grid)

# Flatten for DeepXDE
phi_ex = phi_ex_mesh.flatten()[:, None]
rho_ex = rho_ex_mesh.flatten()[:, None]

# Add noise
phi_obs = add_noise(phi_ex, NOISE_LEVEL, SEED)

In [ ]:
# =============================================================================
# Train Both Models
# =============================================================================

# Train Poisson-only model
phi_pred_P, rho_pred_P, loss_P = train_model('Poisson', phi_obs, XT_data, SEED)

# Train Coupled model (Poisson + Diffusion-Decay)
phi_pred_C, rho_pred_C, loss_C = train_model('Coupled', phi_obs, XT_data, SEED)

In [ ]:
# =============================================================================
# Compute Metrics
# =============================================================================

def relative_L2_error(pred, exact):
    """Relative L2 error: ||pred - exact||₂ / ||exact||₂"""
    return np.sqrt(np.mean((pred - exact)**2)) / np.sqrt(np.mean(exact**2))

# Poisson model metrics
phi_L2_P = relative_L2_error(phi_pred_P, phi_ex)
rho_L2_P = relative_L2_error(rho_pred_P, rho_ex)

# Coupled model metrics
phi_L2_C = relative_L2_error(phi_pred_C, phi_ex)
rho_L2_C = relative_L2_error(rho_pred_C, rho_ex)

print("\n" + "="*50)
print(f"Results (Gaussian, noise = {NOISE_LEVEL*100:.0f}%)")
print("="*50)
print(f"{'Model':<12} {'φ L2 Error':<15} {'ρ L2 Error':<15}")
print("-"*50)
print(f"{'Poisson':<12} {phi_L2_P*100:>10.2f}%     {rho_L2_P*100:>10.2f}%")
print(f"{'Coupled':<12} {phi_L2_C*100:>10.2f}%     {rho_L2_C*100:>10.2f}%")
print("="*50)
print(f"\nρ error reduction: {(1 - rho_L2_C/rho_L2_P)*100:.1f}%")

In [ ]:
# =============================================================================
# Setup for plotting
# =============================================================================
def reshape_to_grid(arr):
    return arr.reshape(NUM_X, NUM_T)

phi_ex_grid = reshape_to_grid(phi_ex)
rho_ex_grid = reshape_to_grid(rho_ex)
phi_obs_grid = reshape_to_grid(phi_obs)
rho_pred_P_grid = reshape_to_grid(rho_pred_P)
rho_pred_C_grid = reshape_to_grid(rho_pred_C)

import matplotlib as mpl
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['STIXGeneral'],
    'mathtext.fontset': 'stix',
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'axes.linewidth': 0.8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.fontsize': 7,
    'legend.frameon': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
})

DOUBLE_COL = 6.69
C_TRUE, C_P, C_C = '#000000', '#D55E00', '#0072B2'
t_indices = [0, 25, 50, 75, 100]
t_values = [0, 0.25, 0.5, 0.75, 1.0]

In [ ]:
# =============================================================================
# Figure 1
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL, DOUBLE_COL / 3))

# (a) Noisy observation
ax = axes[0]
ax.plot(x_grid, phi_ex_grid[:, 0], '-', color=C_TRUE, lw=1.0, label=r'$\phi_{ex}$')
ax.scatter(x_grid[::2], reshape_to_grid(phi_obs)[::2, 0],
           c=C_P, s=8, alpha=0.6, label=r'$\phi_{obs}$', zorder=5)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$\phi$')
ax.set_title(f'(a) Noisy data ($\\sigma$={int(NOISE_LEVEL*100)}%)')
ax.set_xlim(-1, 1)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect(2/3)
ax.legend(loc='lower right', fontsize=5)

# (b) ρ comparison at t=1
ax = axes[1]
ax.plot(x_grid, rho_ex_grid[:, -1], '-', color=C_TRUE, lw=1.0, label='True')
ax.plot(x_grid, rho_pred_P_grid[:, -1], '--', color=C_P, lw=1.0, label='Poisson')
ax.plot(x_grid, rho_pred_C_grid[:, -1], '-', color=C_C, lw=1.0, label='Coupled')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$\rho$')
ax.set_title('(b) Reconstruction at $t=1$')
ax.set_xlim(-1, 1)
ax.set_ylim(-6, 6)
ax.set_aspect(2/12)
ax.legend(loc='upper right', fontsize=5)

# (c) L2 error bar chart
ax = axes[2]
L2_P_list, L2_C_list = [], []
for t_idx in t_indices:
    rho_ex_slice = rho_ex_grid[:, t_idx]
    L2_P = np.sqrt(np.mean((rho_pred_P_grid[:, t_idx] - rho_ex_slice)**2)) / np.sqrt(np.mean(rho_ex_slice**2))
    L2_C = np.sqrt(np.mean((rho_pred_C_grid[:, t_idx] - rho_ex_slice)**2)) / np.sqrt(np.mean(rho_ex_slice**2))
    L2_P_list.append(L2_P * 100)
    L2_C_list.append(L2_C * 100)

x_pos = np.arange(len(t_values))
width = 0.35
ax.bar(x_pos - width/2, L2_P_list, width, color=C_P, label='Poisson')
ax.bar(x_pos + width/2, L2_C_list, width, color=C_C, label='Coupled')
ax.set_xlabel(r'$t$')
ax.set_ylabel(r'Relative $L_2$ Error (%)')
ax.set_title('(c) Error comparison')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(t) for t in t_values])
ax.set_ylim(0, max(L2_P_list) * 1.2)
ax.legend(loc='upper left', fontsize=5)

fig.tight_layout()
plt.savefig('demo_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# Figure 2
# =============================================================================
fig, axes = plt.subplots(2, 5, figsize=(DOUBLE_COL, DOUBLE_COL * 0.42))

for i, (t_idx, t_val) in enumerate(zip(t_indices, t_values)):
    # Top: φ
    ax = axes[0, i]
    ax.plot(x_grid, phi_ex_grid[:, t_idx], '-', color=C_TRUE, lw=0.8, label='True')
    ax.plot(x_grid, reshape_to_grid(phi_pred_P)[:, t_idx], '-', color=C_P, lw=0.8, label='Poisson')
    ax.plot(x_grid, reshape_to_grid(phi_pred_C)[:, t_idx], '-', color=C_C, lw=0.8, label='Coupled')
    ax.set_title(f'$t = {t_val}$')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1.2, 1.2)
    ax.set_xlabel(r'$x$')
    if i == 0:
        ax.set_ylabel(r'$\phi$')
        ax.legend(loc='lower right', fontsize=4)

    # Bottom: ρ
    ax = axes[1, i]
    ax.plot(x_grid, rho_ex_grid[:, t_idx], '-', color=C_TRUE, lw=0.8)
    ax.plot(x_grid, rho_pred_P_grid[:, t_idx], '-', color=C_P, lw=0.8)
    ax.plot(x_grid, rho_pred_C_grid[:, t_idx], '-', color=C_C, lw=0.8)
    ax.set_xlabel(r'$x$')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-11, 11)
    if i == 0:
        ax.set_ylabel(r'$\rho$')

fig.tight_layout()
fig.text(0.5, -0.02, rf'Noise level: $\sigma = {int(NOISE_LEVEL*100)}\%$',
         ha='center', va='top', fontsize=7)
plt.savefig('demo_evolution.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

This demo shows that for **Gaussian initial conditions** (realistic KPFM charge injection):

1. **Poisson** model amplifies noise, especially near the peak
2. **Coupled** model provides stable reconstruction of the spreading Gaussian

The diffusion-decay equation correctly captures:
- **Spreading**: σ(t) = √(σ₀² + 2Dt)
- **Decay**: Q(t) = Q₀ exp(-kt)

---

### Model Naming Convention

| Code | Paper | Description |
|------|-------|-------------|
| `Poisson` | P-iPINN | Poisson-only inverse PINN |
| `Coupled` | PD-iPINN | Poisson + Diffusion-Decay |